# Notebook 15 — TinyDETR: Transformer-Based Object Detection

## Learning Objectives
- Understand DETR's set-prediction paradigm: N queries → N box predictions.
- Build and train `TinyDETR` combining a CNN backbone + Transformer encoder + decoder.
- Understand the greedy IoU matcher (simplified version of Hungarian matching).
- Visualise object query activations and their spatial attention.
- Compare TinyDETR vs TinyGridDetector: no NMS, no anchors, but needs more epochs.
- Explain why DETR eliminates the need for NMS and anchor boxes.

## Big Picture
DETR (DEtection TRansformer, Carion et al. 2020) reimagined object detection as a set prediction
problem. Instead of a grid of predictions or anchor boxes, a fixed set of $N$ learnable *object
queries* each decode one potential object from the encoded image features. Each query attends to
the full image and produces exactly one class + box prediction. No NMS required.

## Dataset
**SyntheticShapes (multi_detection mode)** — same dataset as Notebook 14.

## Architecture
```
Input [B, 3, 128, 128]
  └─ TinyCNNBackbone (stride=8)  → [B, 128, 16, 16]
  └─ Conv1x1(128→64)             → [B, 64,  16, 16]  (project to d_model)
  └─ Flatten + Transpose         → [B, 256, 64]  (256 visual tokens)
  └─ + SinusoidalPositionalEncoding
  └─ TransformerEncoder (2 blocks) → [B, 256, 64]  (memory)
  └─ N=5 learnable object queries  → [B, 5, 64]
  └─ TransformerDecoder (2 blocks, queries attend to memory)
                                   → [B, 5, 64]  (decoded queries)
  ├─ Class head: Linear(64→4)      → class_logits [B, 5, 4]  (3 cls + 1 no-object)
  └─ Box head:   MLP(64→128→4)→Sig → boxes [B, 5, 4]  in normalised cxcywh
```

## Theory
**Object queries** are learnable vectors, one per potential object slot. They are initialised
randomly and learned during training. Each query specialises in detecting objects in different
positions or of different sizes.

**Cross-attention in decoder**: each query attends to the encoded visual memory tokens.
- Q = object queries
- K, V = encoded visual tokens (memory)
- The query can 'look at' any region of the feature map to find its object.

**Set prediction loss**: there is no pre-defined assignment between queries and GT objects.
We use a greedy IoU matcher to assign each GT box to the best-matching query. Unmatched
queries predict the 'no-object' class (lower-weighted in the loss).

**Why no NMS?** Each query is trained to predict *at most one* object. With $N$ queries and $M$
objects ($N > M$), the model learns to assign each GT to a unique query and predict
'no-object' for the remaining $N - M$ queries.

**Reference**: Carion et al., *End-to-End Object Detection with Transformers*, ECCV 2020.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import csv
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary, save_table_csv
from src.utils.paths import RUNS_DETECTION_DIR
from src.data.synthetic_shapes import load_shapes_data, multi_detection_collate_fn
from src.models.tiny_detr import TinyDETR
from src.losses.detection_losses import tiny_detr_loss
from src.metrics.detection import box_cxcywh_to_xyxy, box_iou
from src.visualization.detection import draw_bounding_boxes, plot_detection_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_DETECTION_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

## Dataset
### 2. Load Synthetic Shapes (multi_detection mode)

In [ ]:
IMAGE_SIZE         = 128
NUM_CLASSES        = 3     # circle, square, triangle
NUM_QUERIES        = 5     # max detectable objects per image
D_MODEL            = 64
NUM_HEADS          = 4
NUM_ENC_LAYERS     = 2
NUM_DEC_LAYERS     = 2
DIM_FF             = 128
MAX_OBJECTS        = 3
BATCH_SIZE         = 16
NUM_EPOCHS         = 10
LR                 = 1e-3
LAMBDA_CLASS       = 1.0
LAMBDA_BOX         = 5.0
LAMBDA_NO_OBJECT   = 0.2
CLASS_NAMES        = ['circle', 'square', 'triangle']

train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='multi_detection',
    max_objects=MAX_OBJECTS,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, class_ids_list, boxes_list = next(iter(train_loader))
print(f'Image batch shape   : {imgs.shape}')
print(f'Max objects per img : {MAX_OBJECTS}  |  Num queries: {NUM_QUERIES}')
print(f'Example: {len(class_ids_list[0])} objects in first image')

### 3. Build TinyDETR

In [ ]:
model = TinyDETR(
    in_channels=3,
    num_classes=NUM_CLASSES,
    num_queries=NUM_QUERIES,
    image_size=IMAGE_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_encoder_layers=NUM_ENC_LAYERS,
    num_decoder_layers=NUM_DEC_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=0.1,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters   : {n_params:,}')
print(f'No-object class index  : {model.no_object_class}  (last class = no-object)')

# Shape trace
x_sample = imgs[:2].to(device)                     # [2, 3, 128, 128]
with torch.no_grad():
    cls_logits, pred_boxes = model(x_sample)

print(f'Input shape       : {x_sample.shape}')
print(f'class_logits shape: {cls_logits.shape}  (B, num_queries, num_classes+1)')
print(f'pred_boxes shape  : {pred_boxes.shape}   (B, num_queries, 4) in normalised cxcywh')
print(f'Box range         : [{pred_boxes.min():.3f}, {pred_boxes.max():.3f}]  (sigmoid)')

## Sanity Checks

In [ ]:
# 1. Output shapes
assert cls_logits.shape == (2, NUM_QUERIES, NUM_CLASSES + 1)
assert pred_boxes.shape == (2, NUM_QUERIES, 4)
print('Output shape checks passed.')

# 2. Boxes in [0, 1]
assert pred_boxes.min() >= 0 and pred_boxes.max() <= 1
print('Boxes in [0, 1] — check!')

# 3. No NaN
assert not torch.isnan(cls_logits).any() and not torch.isnan(pred_boxes).any()
print('No NaN in output.')

# 4. With random init, most queries predict no-object (max logit at no-object index)
predicted_classes = cls_logits.argmax(-1)  # [2, num_queries]
no_obj_frac = (predicted_classes == model.no_object_class).float().mean().item()
print(f'Fraction predicting no-object (random init): {no_obj_frac:.2f}')

# 5. Loss computable
test_loss = tiny_detr_loss(
    cls_logits, pred_boxes,
    class_ids_list[:2], [b.to(device) for b in boxes_list[:2]],
    no_object_class=model.no_object_class,
    lambda_class=LAMBDA_CLASS, lambda_box=LAMBDA_BOX, lambda_no_object=LAMBDA_NO_OBJECT,
    device=device,
)
print(f'Initial DETR loss: {test_loss.item():.4f}')
print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_cls_list, batch_boxes_list in train_loader:
        batch_imgs      = batch_imgs.to(device)                   # [B, 3, H, W]
        batch_boxes_dev = [b.to(device) for b in batch_boxes_list]
        optimizer.zero_grad()
        cls_logits, pred_boxes = model(batch_imgs)                # [B,Q,C+1], [B,Q,4]
        loss = tiny_detr_loss(
            cls_logits, pred_boxes,
            batch_cls_list, batch_boxes_dev,
            no_object_class=model.no_object_class,
            lambda_class=LAMBDA_CLASS, lambda_box=LAMBDA_BOX,
            lambda_no_object=LAMBDA_NO_OBJECT,
            device=device,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_n = 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_cls_list, batch_boxes_list in val_loader:
            batch_imgs      = batch_imgs.to(device)
            batch_boxes_dev = [b.to(device) for b in batch_boxes_list]
            cls_logits, pred_boxes = model(batch_imgs)
            v_loss += tiny_detr_loss(
                cls_logits, pred_boxes,
                batch_cls_list, batch_boxes_dev,
                no_object_class=model.no_object_class,
                lambda_class=LAMBDA_CLASS, lambda_box=LAMBDA_BOX,
                lambda_no_object=LAMBDA_NO_OBJECT,
                device=device,
            ).item()
            v_n += 1
    history['val_loss'].append(v_loss / v_n)
    scheduler.step()

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
# Evaluate: for each image, keep only queries predicting foreground classes (not no-object)
model.eval()
total_tp, total_pred, total_gt = 0, 0, 0

with torch.no_grad():
    for batch_imgs, batch_cls_list, batch_boxes_list in val_loader:
        batch_imgs = batch_imgs.to(device)
        cls_logits, pred_boxes = model(batch_imgs)  # [B, Q, C+1], [B, Q, 4]
        B = batch_imgs.size(0)

        for b in range(B):
            # Filter out no-object predictions
            pred_cls = cls_logits[b].argmax(-1)     # [Q]
            keep = pred_cls != model.no_object_class
            pred_boxes_b = pred_boxes[b][keep]       # [k, 4]
            gt_boxes_b   = batch_boxes_list[b]       # [M, 4]
            total_pred  += keep.sum().item()
            total_gt    += len(gt_boxes_b)

            if keep.sum() == 0 or len(gt_boxes_b) == 0:
                continue

            pred_xyxy = box_cxcywh_to_xyxy(pred_boxes_b)
            gt_xyxy   = box_cxcywh_to_xyxy(gt_boxes_b.to(device))
            iou_mat   = box_iou(pred_xyxy, gt_xyxy)
            matched = set()
            for pi in range(len(pred_boxes_b)):
                for gi in range(len(gt_boxes_b)):
                    if gi not in matched and iou_mat[pi, gi].item() >= 0.5:
                        total_tp += 1
                        matched.add(gi)
                        break

precision = total_tp / max(total_pred, 1)
recall    = total_tp / max(total_gt, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-8)

print(f'Precision @IoU0.5 : {precision:.4f}')
print(f'Recall    @IoU0.5 : {recall:.4f}')
print(f'F1 Score          : {f1:.4f}')
print(f'GT objects: {total_gt}  |  Predicted: {total_pred}  |  TPs: {total_tp}')

metrics = {
    'precision_at_iou50': float(precision),
    'recall_at_iou50': float(recall),
    'f1_score': float(f1),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'num_queries': NUM_QUERIES,
}
save_metrics_json(metrics, run_dir / 'tiny_detr_metrics.json')
print('Metrics saved.')

## Visualization

In [ ]:
# Training curves
from src.visualization.plots import plot_training_curves
plot_training_curves(
    history,
    save_path=run_dir / 'tiny_detr_training_curve.png',
    title='TinyDETR Training Curves',
)
print('Training curve saved.')

In [ ]:
# Object query activation analysis
# For each query, find which class it most commonly predicts across validation set
model.eval()
query_class_counts = torch.zeros(NUM_QUERIES, NUM_CLASSES + 1)  # +1 for no-object

with torch.no_grad():
    for batch_imgs, _, _ in val_loader:
        cls_logits, _ = model(batch_imgs.to(device))  # [B, Q, C+1]
        pred_cls = cls_logits.argmax(-1)               # [B, Q]
        for q in range(NUM_QUERIES):
            for c in range(NUM_CLASSES + 1):
                query_class_counts[q, c] += (pred_cls[:, q] == c).sum().item()

# Save query table CSV
class_labels = CLASS_NAMES + ['no-object']
query_rows = []
for q in range(NUM_QUERIES):
    total = query_class_counts[q].sum().item()
    most_common = class_labels[query_class_counts[q].argmax().item()]
    row = {'query': q, 'most_common_class': most_common}
    for c, cname in enumerate(class_labels):
        row[f'pct_{cname}'] = float(query_class_counts[q, c] / total * 100)
    query_rows.append(row)

save_table_csv(query_rows, run_dir / 'tiny_detr_query_table.csv')
print('Query table:')
print(f'{"Query":<8} {"Most common":<15} {"circle%":>10} {"square%":>10} {"triangle%":>12} {"no-obj%":>10}')
for r in query_rows:
    print(f'{r["query"]:<8} {r["most_common_class"]:<15} '
          f'{r["pct_circle"]:>10.1f} {r["pct_square"]:>10.1f} '
          f'{r["pct_triangle"]:>12.1f} {r["pct_no-object"]:>10.1f}')

In [ ]:
# Object query class distribution heatmap
fig, ax = plt.subplots(figsize=(8, 4))
class_labels_display = CLASS_NAMES + ['no-object']
query_pcts = query_class_counts / query_class_counts.sum(dim=1, keepdim=True) * 100
im = ax.imshow(query_pcts.T.numpy(), cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')
plt.colorbar(im, ax=ax, label='% of predictions')
ax.set_xticks(range(NUM_QUERIES))
ax.set_xticklabels([f'Q{q}' for q in range(NUM_QUERIES)])
ax.set_yticks(range(NUM_CLASSES + 1))
ax.set_yticklabels(class_labels_display)
ax.set_xlabel('Object Query')
ax.set_ylabel('Predicted Class')
ax.set_title('Object Query Class Distribution (% of val predictions)')
for q in range(NUM_QUERIES):
    for c in range(NUM_CLASSES + 1):
        ax.text(q, c, f'{query_pcts[q, c]:.0f}', ha='center', va='center',
                fontsize=9, color='black' if query_pcts[q, c] < 60 else 'white')
plt.tight_layout()
fig.savefig(run_dir / 'tiny_detr_query_heatmap.png', dpi=120)
plt.close(fig)
print('Query heatmap saved.')

In [ ]:
# Predicted boxes visualisation
model.eval()
vis_imgs, vis_cls_list, vis_boxes_list = next(iter(val_loader))
with torch.no_grad():
    vis_cls_logits, vis_pred_boxes = model(vis_imgs[:4].to(device))

pred_boxes_px_list, pred_cls_list, gt_boxes_px_list, gt_cls_list = [], [], [], []
for i in range(4):
    pred_cls_i = vis_cls_logits[i].argmax(-1)           # [Q]
    keep = pred_cls_i != model.no_object_class           # keep foreground queries
    pb = (box_cxcywh_to_xyxy(vis_pred_boxes[i][keep]).cpu() * IMAGE_SIZE).numpy() if keep.any() else np.zeros((0, 4))
    gb = (box_cxcywh_to_xyxy(vis_boxes_list[i]) * IMAGE_SIZE).numpy()
    pred_boxes_px_list.append(pb)
    pred_cls_list.append(pred_cls_i[keep].cpu().tolist())
    gt_boxes_px_list.append(gb)
    gt_cls_list.append(vis_cls_list[i])

plot_detection_examples(
    images=vis_imgs[:4].numpy(),
    gt_boxes_list=gt_boxes_px_list,
    gt_class_ids_list=gt_cls_list,
    pred_boxes_list=pred_boxes_px_list,
    pred_class_ids_list=pred_cls_list,
    save_path=run_dir / 'tiny_detr_predictions.png',
    title='TinyDETR: GT (left) vs Predicted (right) — no NMS needed',
    n_examples=4,
    class_names=CLASS_NAMES,
)
print('Prediction examples saved.')

## Failure Cases

**Why TinyDETR may underperform the grid detector on small datasets:**

1. **Slow convergence**: DETR famously needs 500 epochs on COCO. With only 10 epochs here,
   the bipartite matching and attention patterns have not fully converged. The grid detector
   converges faster due to its hard spatial assignment.

2. **Greedy vs optimal matching**: Real DETR uses the Hungarian algorithm for optimal assignment.
   Our greedy IoU matcher is an approximation that may give suboptimal training signals.

3. **All queries predict no-object**: If the class loss weight for no-object (`lambda_no_object`)
   is too high, the model collapses to predicting no-object for all queries.

4. **Query specialisation not learned**: With few epochs, different queries may not yet
   specialise for different positions/classes. Check the query heatmap — if all queries
   show similar distributions, more training is needed.

5. **Gradient clipping sensitivity**: DETR's Transformer decoder needs careful gradient clipping.
   Too large → unstable; too small → slow convergence. We use clip=0.1.

## Exercises

1. **More epochs**: Train for 30 epochs instead of 10. Does recall improve significantly?
   Plot the learning curve and identify when queries start specialising.

2. **Hungarian matching**: Implement `scipy.optimize.linear_sum_assignment` as the matcher.
   Replace `simple_iou_matcher`. Does the optimal assignment improve F1 vs greedy?

3. **Query position embedding**: Add a learned 2D positional embedding to the object queries
   (each query gets a spatial prior). Does this improve convergence speed?

4. **Decoder self-attention**: Add decoder self-attention (queries attend to each other before
   cross-attending to memory). This is part of the original DETR. How does it affect duplicate
   predictions?

5. **Grid vs DETR comparison**: For each of the 200 validation images, record the number of
   correct detections (IoU≥0.5) for TinyGridDetector vs TinyDETR. On which image types does
   DETR outperform the grid detector?

## Key Takeaways

- **Object queries** are the key innovation: learnable vectors that attend to visual features
  and each decode one potential object — no anchors, no grid assignments.
- **Cross-attention** lets each query look at any part of the feature map, enabling flexible
  and global spatial reasoning.
- **Set prediction + matching loss** replaces the complex anchor-assignment + NMS pipeline:
  the model learns to assign each GT to exactly one query.
- **No NMS required**: queries are trained to be non-duplicate by design.
- **Slow convergence** is DETR's known weakness: the attention patterns need many epochs to
  converge, especially without pre-training.
- Modern extensions (Deformable-DETR, DAB-DETR, DN-DETR) address the slow convergence by
  adding deformable attention, dynamic anchors, and denoising training.

## Saved Outputs

In [ ]:
save_markdown_report(
    title='TinyDETR — Transformer-Based Object Detection',
    summary=(
        f'TinyDETR ({NUM_QUERIES} queries, d_model={D_MODEL}, enc={NUM_ENC_LAYERS}, dec={NUM_DEC_LAYERS}) '
        f'trained {NUM_EPOCHS} epochs. '
        f'Precision@IoU0.5: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}.'
    ),
    metrics=metrics,
    figures=['tiny_detr_training_curve.png', 'tiny_detr_predictions.png',
             'tiny_detr_query_heatmap.png'],
    tables=['tiny_detr_query_table.csv'],
    output_path=run_dir / 'tiny_detr_report.md',
    device=str(device),
    hyperparams={
        'num_queries': NUM_QUERIES, 'd_model': D_MODEL,
        'num_enc_layers': NUM_ENC_LAYERS, 'num_dec_layers': NUM_DEC_LAYERS,
        'lambda_box': LAMBDA_BOX, 'lambda_no_object': LAMBDA_NO_OBJECT,
        'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR,
    },
    architecture='CNNBackbone(s=8) + proj + SinPE + TransformerEncoder×2 + N_query → TransformerDecoder×2 → ClassHead + BoxHead',
    loss_fn='CE(class, weighted no-object) + lambda_box * SmoothL1(matched boxes)',
)

update_runs_summary(
    session_name='tiny_detr',
    report_path=run_dir / 'tiny_detr_report.md',
    metrics=metrics,
    figures=['tiny_detr_training_curve.png', 'tiny_detr_predictions.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/tiny_detr_metrics.json')
print(f'  {run_dir}/tiny_detr_training_curve.png')
print(f'  {run_dir}/tiny_detr_predictions.png')
print(f'  {run_dir}/tiny_detr_query_table.csv')
print(f'  {run_dir}/tiny_detr_report.md')